# DALI 蛋白质结构比对工作流

这个 notebook 支持使用**在线 DALI 服务器**或**本地 DALI 安装**进行结构比对并保存结果。

**新特性：**
- ✨ 支持在线 DALI 服务器 (ekhidna2.biocenter.helsinki.fi)
- 🔄 自动模式：优先使用在线服务器，失败时自动回退到本地
- 📦 模块化：使用 `protflow.prediction.dali` 模块
- 🚀 批处理：高效处理多个结构
- 📊 结果分析：自动保存和可视化

## 目标

1. 使用**在线或本地 DALI** 对查询结构进行比对
2. 支持三种模式：
   - `online`: 使用在线 DALI 服务器
   - `local`: 使用本地 DALI 安装
   - `auto`: 自动选择（推荐）
3. 批处理多个 PDB 文件
4. 将结果转换成可视化或下游分析需要的格式

## 前提条件

- Python 3.10+（服务器已安装）
- **在线模式**: 需要网络访问 `ekhidna2.biocenter.helsinki.fi`
- **本地模式**: DALI 路径（例如 `dali.pl` 脚本）在 `PATH` 中或设置为绝对路径
- 结构文件统一放在 `data/structures/` 下，支持 `.pdb`、`.cif`、`.ent` 格式
- 已安装 `protflow` 包：`pip install -e .`

## DALI 数据库说明

### 在线模式
- 自动使用 DALI 在线服务器的最新数据库
- 支持的数据库：`pdb25`, `pdb50`, `pdb90`, `pdb100`
- 无需本地存储空间
- 网络连接要求稳定

### 本地模式
- 本地 DALI 安装默认依赖 `pdb100` 或 `pdb90` 数据库
- 请确保磁盘有 ≥50 GB 空间
- 推荐做法是在服务器上运行 `setup_dali_db.sh`（或参考官方 `fetch_dali.pl`）定期拉取最新 PDB 库
- 若仅做本地比对，可把 `DALI_DB_DIR` 环境变量指向挂载盘，然后在命令行参数中加入 `-databank <path>`

### 推荐使用在线模式
- 无需维护本地数据库
- 始终使用最新的 PDB 数据
- 节省磁盘空间

## 工作流程概览

1. 导入 `protflow.prediction.dali` 模块
2. 配置 DALI aligner（选择在线/本地/自动模式）
3. 枚举 query 结构文件
4. 对每个结构运行 DALI 并捕获结果
5. 解析输出，导出排名，必要时可视化

In [ ]:
# 使用共享后端模块进行DALI结构比对（所有业务逻辑在后端）
import sys
from pathlib import Path

# 添加protflow到路径
project_root = Path.cwd()
while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")

# 导入DALI模块
from protflow.prediction.dali import DaliAligner, DaliResult, batch_align, run_dali_alignment
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

print("✓ DALI 比对工具已加载（使用后端模块）")


In [ ]:
ROOT = Path().resolve()
STRUCTURES_DIR = ROOT / "data" / "structures"
OUTPUT_BASE = ROOT / "outputs" / "dali"
ESM3_PRED_DIR = ROOT / "outputs" / "esm3" / "predictions"

# DALI Configuration
DALI_MODE = 'auto'  # 'online', 'local', or 'auto'
DALI_DATABASE = 'pdb25'  # For online mode: pdb25, pdb50, pdb90, pdb100
DALI_CMD = None  # Path to dali.pl for local mode (None = auto-detect)

if not STRUCTURES_DIR.exists():
    STRUCTURES_DIR.mkdir(parents=True, exist_ok=True)
    logging.warning(f"{STRUCTURES_DIR} 已创建，请放入结构文件。")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print(f"配置：")
print(f"  模式: {DALI_MODE}")
print(f"  数据库: {DALI_DATABASE}")
print(f"  结构目录: {STRUCTURES_DIR}")
print(f"  输出目录: {OUTPUT_BASE}")

## 初始化 DALI Aligner

创建 DALI aligner 实例。根据 `DALI_MODE` 配置选择工作模式：
- `online`: 使用在线 DALI 服务器
- `local`: 使用本地 DALI 安装
- `auto`: 自动选择（优先在线，失败时回退到本地）

In [ ]:
# Initialize DALI aligner
aligner = DaliAligner(
    mode=DALI_MODE,
    dali_cmd=DALI_CMD,
    output_dir=OUTPUT_BASE,
    timeout=300,  # 5 minutes for online queries
)

print(f"✓ DALI Aligner 已初始化")
print(f"  实际使用模式将在运行时确定")

## 查询结构准备

列出所有待比对的 PDB/CIF/ENT 文件，确保命名一致，便于批处理。

## ESM3 结果整合（萜合酶探索）

为了锁定潜在萜类合酶，可以把 ESM3 Workflow 的预测结构（PDB）自动同步到 `data/structures/`，然后交给 DALI 做结构相似性搜索。
- 默认假设预测结果保存在 `outputs/esm3/predictions/`，文件名和 `target_id` 一致
- 同步时会为每个文件加上 `esm3_` 前缀，避免覆盖手工准备的结构
- 结合 Z-score、高度保守的活性位点和注释，可以快速筛查疑似萜合酶

In [ ]:
def sync_esm3_predictions(source_dir: Path = ESM3_PRED_DIR, target_dir: Path = STRUCTURES_DIR) -> list[Path]:
    synced = []
    if not source_dir.exists():
        logging.info("ESM3 预测目录 %s 不存在，跳过同步。", source_dir)
        return synced
    target_dir.mkdir(parents=True, exist_ok=True)
    for ext in ("*.pdb", "*.cif"):
        for structure in sorted(source_dir.glob(ext)):
            dest = target_dir / f"esm3_{structure.name}"
            if not dest.exists() or structure.stat().st_mtime > dest.stat().st_mtime:
                shutil.copy2(structure, dest)
                logging.info("同步 %s -> %s", structure.name, dest.name)
            synced.append(dest)
    return synced

In [ ]:
synced_esm3 = sync_esm3_predictions()
query_patterns = ["*.pdb", "*.cif", "*.ent"]
queries: list[Path] = []
for pattern in query_patterns:
    queries.extend(sorted(STRUCTURES_DIR.glob(pattern)))
if synced_esm3:
    logging.info("已加入 %d 个 ESM3 预测结构用于 DALI。", len(synced_esm3))
# 去重保持顺序
unique: list[Path] = []
seen: set[Path] = set()
for path in queries:
    if path not in seen:
        unique.append(path)
        seen.add(path)
queries = unique
if not queries:
    logging.warning("在 %s 中没有发现 PDB/CIF 文件，请先放入结构。", STRUCTURES_DIR)
else:
    print(
        "找到 %d 个结构文件，前 5 个：" % len(queries),
        *[q.name for q in queries[:5]],
        sep="\n",
    )

## DALI 结构比对

使用新的 `DaliAligner` 类进行结构比对。支持：
- 单个结构比对
- 批量处理
- 在线和本地模式自动切换

In [ ]:
# 示例：比对单个结构
if queries:
    sample_query = queries[0]
    print(f"比对示例: {sample_query.name}")
    
    results = aligner.align(
        query_structure=sample_query,
        database=DALI_DATABASE,
    )
    
    print(f"\n找到 {len(results)} 个比对结果")
    if results:
        print(f"\n前 5 个结果：")
        for r in results[:5]:
            print(f"  {r.rank}. {r.target_pdb:8s} Z={r.z_score:6.2f}  RMSD={r.rmsd:5.2f}")
else:
    print("未找到结构文件，请先放入 data/structures/")

## 批量处理和结果汇总

使用 `align_batch` 方法批量处理所有结构，并生成汇总表格。

In [ ]:
# 批量比对所有结构
if queries:
    print(f"批量处理 {len(queries)} 个结构...")
    
    batch_results = aligner.align_batch(
        query_structures=queries,
        database=DALI_DATABASE,
    )
    
    # 创建汇总表
    summary_df = aligner.summarize_results(batch_results, top_n=10)
    
    if summary_df is not None and not summary_df.empty:
        print(f"\n汇总：找到 {len(summary_df)} 个总比对结果")
        display(summary_df.head(20))
        
        # 保存汇总
        summary_path = OUTPUT_BASE / "dali_batch_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        logging.info(f"批量汇总已保存到 {summary_path}")
    else:
        print("没有结果可汇总")
else:
    print("没有结构文件可处理")

## 示例执行

若已经准备好了结构文件，可先跑一条记录“绿灯”全链路。

In [ ]:
if queries:
    sample_query = queries[0]
    target = OUTPUT_BASE / sample_query.stem
    log_path = run_local_dali(sample_query, target)
    print("示例日志：", log_path)
else:
    print("未找到结构文件，请先放入 data/structures。")


## 下一步

### 可视化和分析
- 如果需要可视化，可将 `results` 传给 `nglview`/`py3Dmol`
- 可以根据 Z-score 筛选高相似度结构
- 结合序列比对和功能注释进行综合分析

### 自动化和集成
- 可将批处理包装进 `papermill`/`nbconvert` 生成报告
- 集成到 Prokka → ESM3 → DALI 工作流
- 添加到 CLI 脚本中实现命令行调用

### 模式切换
如果在线模式不可用，只需修改配置：
```python
DALI_MODE = 'local'  # 切换到本地模式
DALI_CMD = Path('/path/to/dali.pl')  # 指定 DALI 命令路径
```

### 高级用法
```python
# 使用 Python API 直接调用
from protflow.prediction.dali import run_dali_alignment

results = run_dali_alignment(
    query_structure=Path('protein.pdb'),
    mode='online',
    database='pdb25',
)
```

*注：此单元格已移除，不再需要添加 protflow 到路径*